In [1]:
from gensim.models import Word2Vec, Doc2Vec
import pandas as pd
import os
import re
import numpy as np
from collections import Counter
import math

In [2]:
def save_vectors(vectors, model_filename, addition=""):
    os.makedirs("poem_embeddings", exist_ok=True)
    base_name = os.path.splitext(model_filename)[0]
    vec_file = os.path.join("poem_embeddings", f"{base_name}{addition}.npy")
    np.save(vec_file, vectors)
    
    return vec_file

In [3]:
data = pd.read_csv('poem_data_with_birth.csv', engine='python')

In [4]:
# Load Shakespeare to know idsfor similarity tests
data[data['Poet']=='William Shakespeare']

,Title,Poet,Poem,Tags,Emotion,PoetBirth,Period
1296,"Sonnet 123: No, Time, thou shalt not boast tha...",William Shakespeare,"No, Time, thou shalt not boast that I do chang...","Living,Time & Brevity,Arts & Sciences",Fear,1564.0,1550-1780
1297,Sonnet 133: Beshrew that heart that makes my h...,William Shakespeare,Beshrew that heart that makes my heart to groa...,"Love,Break-ups & Vexed Love,Unrequited Love,So...",Sadness,1564.0,1550-1780
1298,"Sonnet 139: O, call not me to justify the wrong",William Shakespeare,"O, call not me to justify the wrongThat thy un...","Love,Break-ups & Vexed Love,Unrequited Love",Anger,1564.0,1550-1780
1299,"Sonnet 142: Love is my sin, and thy dear virtu...",William Shakespeare,"Love is my sin, and thy dear virtue hate,Hate ...","Living,Marriage & Companionship,Love,Desire,Re...",Anger,1564.0,1550-1780
1545,Venus and Adonis,William Shakespeare,Even as the sun with purple-colour’d face ...,NaN,Love,1564.0,1550-1780
...,...,...,...,...,...,...,...
13602,Song: “Full fathom five thy father lies”,William Shakespeare,(from The Tempest)\r\r\n\r\r\n\r\r\n\r\r\nFull...,"Living,Death,Nature,Seas, Rivers, & Streams",NaN,1564.0,1550-1780
13603,"Song: “Hark, hark! the lark at heaven's gate s...",William Shakespeare,"(from Cymbeline)\r\r\n\r\r\n\r\r\n\r\r\nHark, ...","Nature,Landscapes & Pastorals",NaN,1564.0,1550-1780
13604,Song: “O Mistress mine where are you roaming?”,William Shakespeare,(from Twelfth Night)\r\r\n\r\r\n\r\r\n\r\r\nO ...,"Living,Time & Brevity,Love,Classic Love,Desire...",NaN,1564.0,1550-1780
13605,Song: “Orpheus with his lute made trees”,William Shakespeare,(from Henry VIII)\r\r\n\r\r\n\r\r\n\r\r\nOrphe...,"Nature,Landscapes & Pastorals,Arts & Sciences,...",NaN,1564.0,1550-1780


# Load stopwords to ignore in word2vec

Keep:
- Negations: not, no, never
- Prepositions: often carry imagery (under, above, beyond)
- Pronouns: I, you, we → important for voice

In [5]:
# Custom stopwords, nltk removes too much
stop_words = {
    'did', 'll', 'mustn', 'haven', 'both', 'all', 'a', 'only', 'same',
    's', 'aren', 'y', 'having', 'own', 'very', 'isn', 'had',
    'once', 'now', 'hasn', 'does', 'hadn',
    'but', 'shouldn', 'just', 'the', 'weren',
    'o', 'because', 'than', 've', 'been',
    'other', 'any', 'again', 'will',
    'wasn', 'itself', 're', 'do',
    'mightn', 'so', 'or',
    'each', 'too', 'needn', 'wouldn',
    'ma', 'an', 'being', 'can', 'won',
    'such', 'and', 'd', 'if', 'should',
    'are', 'am', 'some',
    'doing', 'has', 'its',
    'm', 'few', 'more', 'be',
    'myself', 'that', 'then',
    'most', 'while', 'was', 'is', 'ain'
}

# Construct poem vectors from word2vec

In [6]:
WORD_MODEL_FILENAME = "w2v_d100_w5_sg.model" #in /models

In [7]:
LINESEP_RE = re.compile(r'\r+\n')
WHITESPACE_RE = re.compile(r'\s+')
TOKEN_RE = re.compile(r'<linesep>|\b\w+\b')

def preprocess(text):
    text = text.lower()
    text = LINESEP_RE.sub(' <linesep> ', text)
    text = WHITESPACE_RE.sub(' ', text)
    return TOKEN_RE.findall(text)

poems = data.iloc[:,2]
tokenized_poems = [preprocess(poem) for poem in poems]

### Method 1 - simple averaging

In [8]:
def get_weighted_embedding(words, wv, stop_words, weight_dict=None, norm_by_weights = True):
    """
    Compute a (possibly TF-IDF weighted) average word embedding.

    Returns:
        poem_vector : np.ndarray
            Averaged embedding vector.
        exceptions : list[str]
            Words not found in the embedding model.
        is_blank : bool
            True if no valid words contributed to the vector.
    """
    exceptions = [w for w in words if w not in wv]
    valid_words = [w for w in words if w in wv and w not in stop_words]
    
    weighted_vectors = []
    weights = []
    
    for w in valid_words:
        tf = 1 / len(valid_words)
        weight = 1.0 if weight_dict is None else tf * weight_dict.get(w, 0.0)
        if weight > 0:
            weighted_vectors.append(wv[w] * weight)
            weights.append(weight)
    
    if not weighted_vectors:
        return np.zeros(wv.vector_size), exceptions, True
    
    weighted_vectors = np.array(weighted_vectors)
    weights = np.array(weights)
    
    poem_vector = np.sum(weighted_vectors, axis=0) / (np.sum(weights) if norm_by_weights else 1)
    
    return poem_vector, exceptions, False

In [9]:
model_word = Word2Vec.load(f"models/{WORD_MODEL_FILENAME}")

In [10]:
def poem_embedding_from_words(tokenized_poems, model_word, stop_words, weight_dict = None, norm_by_weights = True):
    embeddings_word = []
    exception_count = 0
    hit_count = 0
    for index, poem in enumerate(tokenized_poems):
        vec, exceptions, is_blank = get_weighted_embedding(poem, model_word.wv, stop_words, weight_dict, norm_by_weights)
        exception_count += len(exceptions)
        hit_count += len(poem)-len(exceptions)
        if is_blank:
            print("Null poem", index, poem)
        embeddings_word.append(vec)
        if index % 1000 == 0:
            print(f"{index} poems ready, {exception_count} exceptions so far")
    embeddings_word = np.array(embeddings_word)
    return embeddings_word, exception_count, hit_count
    

In [11]:
embeddings_word, exc_count, hit_count = poem_embedding_from_words(
    tokenized_poems,
    model_word,
    stop_words,
    weight_dict = None)
save_vectors(embeddings_word, WORD_MODEL_FILENAME, "_avg")
print(f"{len(embeddings_word)} vectors generated. Average exception count: {exc_count / len(poems)}. Average hit count: {hit_count / len(poems)}. ")

0 poems ready, 0 exceptions so far
1000 poems ready, 6224 exceptions so far
2000 poems ready, 10626 exceptions so far
Null poem 2721 ['d']
3000 poems ready, 15483 exceptions so far
4000 poems ready, 26581 exceptions so far
5000 poems ready, 33372 exceptions so far
6000 poems ready, 36681 exceptions so far
7000 poems ready, 40167 exceptions so far
8000 poems ready, 45163 exceptions so far
9000 poems ready, 50161 exceptions so far
Null poem 9406 ['217']
10000 poems ready, 55351 exceptions so far
11000 poems ready, 63750 exceptions so far
12000 poems ready, 70685 exceptions so far
13000 poems ready, 75526 exceptions so far
13745 vectors generated. Average exception count: 5.769952710076391. Average hit count: 278.3765732993816. 


In [12]:
def get_most_similar_poem_idx(embeddings, base_idx):
    comparison_base = embeddings[base_idx]
    best_sim = -1
    best_idx = -1

    for i, emb in enumerate(embeddings):
        if i == base_idx:
            continue
        norm_base = np.linalg.norm(comparison_base)
        norm_emb = np.linalg.norm(emb)

        # skip zero vectors to avoid divide-by-zero
        if norm_base == 0 or norm_emb == 0:
            continue

        sim = np.dot(comparison_base, emb) / (norm_base * norm_emb)

        if sim > best_sim:
            best_sim = sim
            best_idx = i
    return best_idx, best_sim

In [13]:
base_idx = 1297
best_idx, best_sim = get_most_similar_poem_idx(embeddings_word, base_idx)

print("Most similar index:", best_idx)
print("Similarity:", best_sim)

Most similar index: 1298
Similarity: 0.970015892622509


In [14]:
data.iloc[[base_idx, best_idx]]

,Title,Poet,Poem,Tags,Emotion,PoetBirth,Period
1297,Sonnet 133: Beshrew that heart that makes my h...,William Shakespeare,Beshrew that heart that makes my heart to groa...,"Love,Break-ups & Vexed Love,Unrequited Love,So...",Sadness,1564.0,1550-1780
1298,"Sonnet 139: O, call not me to justify the wrong",William Shakespeare,"O, call not me to justify the wrongThat thy un...","Love,Break-ups & Vexed Love,Unrequited Love",Anger,1564.0,1550-1780


### Method 2 - IDF weighting

In [15]:
def get_idf(tokenized_poems):
    df = Counter()
    for tokens in tokenized_poems:
        unique_words = set(tokens)
        for w in unique_words:
            df[w] += 1

    idf = {
        w: math.log(len(tokenized_poems) / (1 + df[w])) # +1 to avoid zero division
        for w in df
    }
    return idf

idf = get_idf(tokenized_poems)

In [16]:
embeddings_word, exc_count, hit_count = poem_embedding_from_words(
    tokenized_poems,
    model_word,
    stop_words,
    weight_dict = idf)
save_vectors(embeddings_word, WORD_MODEL_FILENAME, "_idf")
print(f"{len(embeddings_word)} vectors generated. Average exception count: {exc_count / len(poems)}. Average hit count: {hit_count / len(poems)}. ")

0 poems ready, 0 exceptions so far
1000 poems ready, 6224 exceptions so far
2000 poems ready, 10626 exceptions so far
Null poem 2721 ['d']
3000 poems ready, 15483 exceptions so far
4000 poems ready, 26581 exceptions so far
5000 poems ready, 33372 exceptions so far
6000 poems ready, 36681 exceptions so far
7000 poems ready, 40167 exceptions so far
8000 poems ready, 45163 exceptions so far
9000 poems ready, 50161 exceptions so far
Null poem 9406 ['217']
10000 poems ready, 55351 exceptions so far
11000 poems ready, 63750 exceptions so far
12000 poems ready, 70685 exceptions so far
13000 poems ready, 75526 exceptions so far
13745 vectors generated. Average exception count: 5.769952710076391. Average hit count: 278.3765732993816. 


In [17]:
base_idx = 1297
best_idx, best_sim = get_most_similar_poem_idx(embeddings_word, base_idx)

print("Most similar index:", best_idx)
print("Similarity:", best_sim)

Most similar index: 2118
Similarity: 0.95600177925523


In [18]:
data.iloc[[base_idx, best_idx]]

,Title,Poet,Poem,Tags,Emotion,PoetBirth,Period
1297,Sonnet 133: Beshrew that heart that makes my h...,William Shakespeare,Beshrew that heart that makes my heart to groa...,"Love,Break-ups & Vexed Love,Unrequited Love,So...",Sadness,1564.0,1550-1780
2118,Night and the Madman,Kahlil Gibran,"“I am like thee, O, Night, dark and naked; I w...",NaN,Fear,1883.0,1901-1940


### Method 3 - SIF weighting

In [19]:
def get_sif(tokenized_poems, a=1e-3):
    word_counts = Counter(
        w for tokens in tokenized_poems for w in tokens
    )
    
    total_tokens = sum(word_counts.values())
    p_w = {
        w: count / total_tokens
        for w, count in word_counts.items()
    }
    
    sif_weights = {
        w: a / (a + p_w[w])
        for w in p_w
    }
    
    return sif_weights

sif = get_sif(tokenized_poems)

In [20]:
embeddings_word, exc_count, hit_count = poem_embedding_from_words(
    tokenized_poems,
    model_word,
    stop_words,
    weight_dict = sif,
    norm_by_weights = False # don't divide by sum of weights here
)

# Extra steps for SIF
# compute mean-centered embeddings
mean_vec = embeddings_word.mean(axis=0)
emb_centered = embeddings_word - mean_vec

# compute first principal component using SVD
u, s, vh = np.linalg.svd(emb_centered, full_matrices=False)
first_pc = vh[0]  # shape: (d,)

# remove first principal component
embeddings_sif = embeddings_word - np.outer(embeddings_word @ first_pc, first_pc)

save_vectors(embeddings_sif, WORD_MODEL_FILENAME, "_sif")
print(f"{len(embeddings_sif)} vectors generated. Average exception count: {exc_count / len(poems)}. Average hit count: {hit_count / len(poems)}. ")

0 poems ready, 0 exceptions so far
1000 poems ready, 6224 exceptions so far
2000 poems ready, 10626 exceptions so far
Null poem 2721 ['d']
3000 poems ready, 15483 exceptions so far
4000 poems ready, 26581 exceptions so far
5000 poems ready, 33372 exceptions so far
6000 poems ready, 36681 exceptions so far
7000 poems ready, 40167 exceptions so far
8000 poems ready, 45163 exceptions so far
9000 poems ready, 50161 exceptions so far
Null poem 9406 ['217']
10000 poems ready, 55351 exceptions so far
11000 poems ready, 63750 exceptions so far
12000 poems ready, 70685 exceptions so far
13000 poems ready, 75526 exceptions so far
13745 vectors generated. Average exception count: 5.769952710076391. Average hit count: 278.3765732993816. 


In [21]:
base_idx = 1297
best_idx, best_sim = get_most_similar_poem_idx(embeddings_word, base_idx)

print("Most similar index:", best_idx)
print("Similarity:", best_sim)

Most similar index: 2254
Similarity: 0.9553731492421359


In [22]:
data.iloc[[base_idx, best_idx]]

,Title,Poet,Poem,Tags,Emotion,PoetBirth,Period
1297,Sonnet 133: Beshrew that heart that makes my h...,William Shakespeare,Beshrew that heart that makes my heart to groa...,"Love,Break-ups & Vexed Love,Unrequited Love,So...",Sadness,1564.0,1550-1780
2254,Sonnet 134: So now I have confessed that he is...,William Shakespeare,"So now I have confessed that he is thine,\r\r\...",NaN,Sadness,1564.0,1550-1780


# Construct poem vectors from line-wise doc2vec
### Simple line averaging method

In [73]:
LINE_MODEL_FILENAME = "d2v_line_d100_w5_dbow.model" #in /models

In [74]:
model_doc = Doc2Vec.load(f"models/{LINE_MODEL_FILENAME}")

In [75]:
def get_embedding_by_line(poem, poem_idx, dv):
    '''Return poem vector'''
    vectors = []
    current_line = []
    line_idx = 0
    for token in poem:
        if token == "<linesep>":
            if current_line:  # skip empty lines
                tag = f"{poem_idx}_{line_idx}"
                vectors.append(dv[tag])
                current_line = []
                line_idx += 1
        else:
            current_line.append(token)

    if current_line:
        tag = f"{poem_idx}_{line_idx}"
        vectors.append(dv[tag])
    return np.mean(vectors, axis=0)

In [76]:
embeddings_doc = []
for index, poem in enumerate(tokenized_poems):
    vec = get_embedding_by_line(poem, index, model_doc.dv)
    embeddings_doc.append(vec)
    if index % 1000 == 0:
        print(f"{index} poems ready")
embeddings_doc = np.array(embeddings_doc)

save_vectors(embeddings_doc, LINE_MODEL_FILENAME)
print(f"{len(embeddings_doc)} vectors generated.")

0 poems ready
1000 poems ready
2000 poems ready
3000 poems ready
4000 poems ready
5000 poems ready
6000 poems ready
7000 poems ready
8000 poems ready
9000 poems ready
10000 poems ready
11000 poems ready
12000 poems ready
13000 poems ready
13745 vectors generated.


In [78]:
base_idx = 1297
best_idx, best_sim = get_most_similar_poem_idx(embeddings_doc, base_idx)

print("Most similar index:", best_idx)
print("Similarity:", best_sim)

Most similar index: 2254
Similarity: 0.6792715


In [79]:
data.iloc[[base_idx, best_idx]]

,Title,Poet,Poem,Tags,Emotion,PoetBirth,Period
1297,Sonnet 133: Beshrew that heart that makes my h...,William Shakespeare,Beshrew that heart that makes my heart to groa...,"Love,Break-ups & Vexed Love,Unrequited Love,So...",Sadness,1564.0,1550-1780
2254,Sonnet 134: So now I have confessed that he is...,William Shakespeare,"So now I have confessed that he is thine,\r\r\...",NaN,Sadness,1564.0,1550-1780


# Save doc2vec full poem embeddings

In [68]:
POEM_MODEL_FILENAME = "d2v_poem_d100_w5_dm.model" #in /models

In [69]:
model_poem = Doc2Vec.load(f"models/{POEM_MODEL_FILENAME}")

In [70]:
embeddings_poem = []
for index, poem in enumerate(tokenized_poems):
    vec = model_poem.dv[index]
    embeddings_poem.append(vec)
    if index % 1000 == 0:
        print(f"{index} poems ready")
embeddings_poem = np.array(embeddings_poem)

save_vectors(embeddings_poem, POEM_MODEL_FILENAME)
print(f"{len(embeddings_poem)} vectors generated.")

0 poems ready
1000 poems ready
2000 poems ready
3000 poems ready
4000 poems ready
5000 poems ready
6000 poems ready
7000 poems ready
8000 poems ready
9000 poems ready
10000 poems ready
11000 poems ready
12000 poems ready
13000 poems ready
13745 vectors generated.


In [84]:
base_idx = 1297
best_idx, best_sim = get_most_similar_poem_idx(embeddings_poem, base_idx)

print("Most similar index:", best_idx)
print("Similarity:", best_sim)

Most similar index: 2254
Similarity: 0.68577653


In [85]:
data.iloc[[base_idx, best_idx]]

,Title,Poet,Poem,Tags,Emotion,PoetBirth,Period
1297,Sonnet 133: Beshrew that heart that makes my h...,William Shakespeare,Beshrew that heart that makes my heart to groa...,"Love,Break-ups & Vexed Love,Unrequited Love,So...",Sadness,1564.0,1550-1780
2254,Sonnet 134: So now I have confessed that he is...,William Shakespeare,"So now I have confessed that he is thine,\r\r\...",NaN,Sadness,1564.0,1550-1780


# Load example

In [62]:
my_vectors = np.load("poem_embeddings/w2v_d100_w5_cbow_avg.npy")
my_vectors.shape

(13745, 100)